# Reading Dolos with J, R, and J++

A hands-on introduction to six matched workspace lenses: three for **Dolos-1.5B**, three for **Qwen2.5-1.5B-Instruct**. We will inspect the top words that each lens reads out at intermediate layers, compare three estimators, and examine the parent/fine-tune pair.

**What these scores are not:** thoughts, guaranteed semantic features, probabilities of an intention, or evidence of deception. The lenses are linear transports of residual activations to the model's unembedding. The examples below are probes, not training or statistical evaluation.

Inspired by the intermediate-variable examples in [Verbalizable Representations Form a Global Workspace in Language Models](https://transformer-circuits.pub/2026/workspace/index.html) and the [J++ quickstart](https://github.com/safety-research/jpp_lens/blob/main/scripts/quickstart.py). We don't assume that a small model faithfully solves each example.

## 0. Setup

**On the training pod:** Start Jupyter with the Python interpreter from `/workspace/dolos/repos/jpp_lens/.venv`, from the J++ repo root. All lens binaries are expected in `/workspace/dolos/artifacts` and the hydrated checkpoints in `/workspace/dolos/models`.

**After publication:** Switch `SOURCE` to `hub` below, and set `HF_REPO` to the actual published repository ID. This downloads the complete Dolos model subfolder and the six lenses; it does not depend on the original weights-only Dolos repository. Install the official J++ code in your environment first.

GPU strongly recommended. The six-lens comparison runs models *sequentially*, not both in memory at once.

In [ ]:
from pathlib import Path
import os, gc, hashlib
import pandas as pd
import torch
from IPython.display import display, Markdown
from workspace_lens import get_hf_model
from workspace_lens.utils import load_lens_file, check_model_matches_config

ROOT = Path("/workspace/dolos")
SOURCE = "local"  # change to "hub" after HF publication
HF_REPO = "bubskee/dolos-qwen25-workspace-lenses"  # proposed; verify release URL first
LAYERS = [0, 4, 8, 12, 16, 20, 24, 26]
TOP_K = 6
print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())

In [ ]:
if SOURCE == "local":
    model_dirs = {"dolos": ROOT / "models/dolos", "parent": ROOT / "models/parent"}
    lens_files = {m: {k: ROOT / "artifacts" / f"{prefix}_{k}_n64.pt" for k in ("j", "r", "jpp")}
                  for m, prefix in (("dolos", "dolos"), ("parent", "parent"))}
elif SOURCE == "hub":
    from huggingface_hub import snapshot_download
    snap = Path(snapshot_download(repo_id=HF_REPO, repo_type="model"))
    # Bundled Dolos; download parent weights independently from its original Hub repository.
    model_dirs = {"dolos": snap / "models/dolos", "parent": "Qwen/Qwen2.5-1.5B-Instruct"}
    lens_files = {m: {k: snap / "lenses" / m / k / "lens.pt" for k in ("j", "r", "jpp")}
                  for m in ("dolos", "parent")}
else:
    raise ValueError("SOURCE must be local or hub")
for m in lens_files:
    for kind, file in lens_files[m].items():
        assert file.is_file(), f"Missing {m}/{kind} lens: {file}"
print("Found all six lens files.")

## 1. Three examples

The first two prompts evoke an **intermediate concept** (heart, Italy) that is not the desired final answer (four, euro). The third does likewise for a spider and its number of legs. These are explanatory examples, not evidence that the model actually computes via these concepts.

In [ ]:
EXAMPLES = {
    "Heart → chambers": "Fact: In humans, the organ that pumps blood through the body has this many chambers: ",
    "Italy → currency": "Fact: The currency used in the country shaped like a boot is",
    "Spider → legs": "Question: How many legs does the animal that spins webs have? Answer:",
}
EXAMPLES

## 2. Load one model and its three lenses

The reader checks lens/config compatibility before applying it. **Use the exact matching checkpoint**; parent and Dolos lenses aren't interchangeable. We reuse one model for all three readouts.

In [ ]:
def load_bundle(model_key):
    model_path = str(model_dirs[model_key])
    model = get_hf_model(model_path, attn_implementation="sdpa")
    lenses = {k: load_lens_file(str(p), hf_model_name=model_path)
              for k, p in lens_files[model_key].items()}
    for kind, lens in lenses.items():
        check_model_matches_config(model, lens.config)
        assert set(LAYERS).issubset(set(lens.source_layers)), (kind, lens.source_layers)
    return model, lenses

def unload(model, lenses):
    del model, lenses
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

model, lenses = load_bundle("dolos")
print("Loaded Dolos lenses:", {k: len(v.source_layers) for k,v in lenses.items()})

## 3. Read out top tokens by layer

The score is obtained by transporting the residual through a **fixed lens map** and applying the model's final unembedding. Rankings below use a simple transparent filter (exclude tokens containing no letters or digits); this is a *tutorial convenience*, not a reproduction of the published evaluation's candidate-preserving readout filter.

In [ ]:
def meaningful_tokens(model):
    # Filter token ids only for display. To compare published recall metrics,
    # use the official evaluation code with its candidate-preserving filter.
    tok = model.tokenizer
    n = model.vocab_size if hasattr(model, "vocab_size") else len(tok)
    excluded = []
    for i in range(n):
        s = tok.decode([i], skip_special_tokens=False)
        if not any(ch.isalnum() for ch in s):
            excluded.append(i)
    return excluded

def inspect(model, lenses, prompt, layers=LAYERS, top_k=TOP_K, excluded=None):
    excluded = meaningful_tokens(model) if excluded is None else excluded
    rows = []
    for kind, lens in lenses.items():
        lens_logits, model_logits, _ = lens.apply(
            model, prompt, layers=layers, token_positions_for_residuals=[-1]
        )
        for layer in layers:
            scores = lens_logits[layer][0].float().clone()
            scores[excluded] = -float("inf")
            ids = scores.topk(top_k).indices.tolist()
            rows.append({"lens": kind.upper(), "layer": layer,
                         "top tokens": " · ".join(repr(model.tokenizer.decode([i])) for i in ids)})
    pred_ids = model_logits[0].float().topk(top_k).indices.tolist()
    print("Prompt:", prompt)
    print("Model next-token candidates:", [model.tokenizer.decode([i]) for i in pred_ids])
    return pd.DataFrame(rows)

excluded = meaningful_tokens(model)
display(inspect(model, lenses, EXAMPLES["Heart → chambers"], excluded=excluded))

In [ ]:
for title in ("Italy → currency", "Spider → legs"):
    display(Markdown(f"### {title}"))
    display(inspect(model, lenses, EXAMPLES[title], excluded=excluded))

## 4. Why a lens result isn't a behavioral result

Look for an intermediate word appearing at earlier layers in J++ than J or R. But don't cherry-pick: a word like `heart` can be highly ranked because of word associations, even if it isn't causally used to arrive at the answer. Likewise, a strong next-token prediction isn't proof the intermediate was explicitly represented. The six-lens release contains held-out aggregate evaluations; this notebook instead presents qualitative readouts.

## 5. Compare to the parent checkpoint

Repeat the same prompt using parent lenses on the parent model. The prompt stays fixed, but the fitted maps and weights change. This produces **descriptive** differences, not a controlled causal account of fine-tuning. Note that the formal parent and Dolos held-out evaluations use separately correctness-filtered populations.

In [ ]:
# Run sequentially to fit comfortably on one GPU.
dolos_example = inspect(model, lenses, EXAMPLES["Heart → chambers"], excluded=excluded)
# Explicitly release references before loading parent.
del model, lenses, excluded
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
parent_model, parent_lenses = load_bundle("parent")
parent_excluded = meaningful_tokens(parent_model)
parent_example = inspect(parent_model, parent_lenses, EXAMPLES["Heart → chambers"], excluded=parent_excluded)
display(Markdown("**Dolos**")); display(dolos_example)
display(Markdown("**Parent Qwen2.5**")); display(parent_example)

## 6. Try your own prompt

A useful probe makes the *bridging variable* distinguishable from the final answer. Try changing the phrasing, add a distractor, or compare two prompts that differ in one substantive fact. Do not infer hidden objectives from isolated top-token lists.

In [ ]:
MY_PROMPT = "Fact: The number of legs on the animal that spins webs is"
display(inspect(parent_model, parent_lenses, MY_PROMPT, excluded=parent_excluded))

## 7. Further reading

- [Original workspace/J-space paper](https://transformer-circuits.pub/2026/workspace/index.html)
- [J++ code and explanations](https://github.com/safety-research/jpp_lens)
- [Dolos six-lens methodology](https://github.com/bfaber-centaur/dolos-lenses/blob/main/docs/DESIGN.md)

**Reproducibility:** For numerical scores and disjoint calibration/held-out samples, use the release metadata and the official `jpp_cli.py evaluate` procedure; the tutorial's token display filter is not the paper's formal scoring protocol.